<a href="https://colab.research.google.com/github/Besendnju/Advanced-analysis-/blob/main/Copy_of_Coastal_Flood_Susceptibility_GeoAI_FIXED_executed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Coastal / Inland Flood Susceptibility — Robust Python GeoAI Pipeline

This version is a corrected, defensive rebuild of the supplied notebook. It preserves the intended workflow—GEE exports → raster harmonization → validation → spatial CV → GeoAI → interpretation → benchmark → scenario statistics → sensitivity/uncertainty—but removes failure-prone assumptions.

**Important:** this notebook does not invent missing results. Missing inputs cause the relevant section to be skipped and recorded in the manifest. A small synthetic smoke test is included so the core raster/model functions can be checked before using the real GEE exports.

## 0. Installation and imports

In Google Colab, install the packages only if they are not already present. The installation line is intentionally commented out so a normal rerun does not repeatedly modify the environment.

In [ ]:
# Colab only, if needed:
# !pip -q install rasterio rioxarray geopandas openpyxl scikit-learn statsmodels xgboost shap seaborn pyproj nbformat

import os, json, warnings, math, sys, platform
from pathlib import Path

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject
from rasterio.transform import Affine
import geopandas as gpd
from pyproj import Geod

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve,
    average_precision_score, confusion_matrix,
    precision_score, recall_score, f1_score, matthews_corrcoef
)
from sklearn.model_selection import GroupKFold
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings("ignore")

try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False

try:
    import shap
    SHAP_AVAILABLE = True
except Exception:
    SHAP_AVAILABLE = False

try:
    from lightgbm import LGBMClassifier
    LGBM_AVAILABLE = True
except Exception:
    LGBM_AVAILABLE = False

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__, "| pandas:", pd.__version__, "| rasterio:", rasterio.__version__)
print("XGBoost:", XGB_AVAILABLE, "| LightGBM:", LGBM_AVAILABLE, "| SHAP:", SHAP_AVAILABLE)

Python: 3.13.15
NumPy: 2.1.3 | pandas: 2.2.3 | rasterio: 1.5.1
XGBoost: True | LightGBM: True | SHAP: True


## 1. Project configuration

The path logic works in Colab/Google Drive or local Windows/Linux execution. Change only `EXPORT_DIR` if your GeoTIFFs are elsewhere.

In [ ]:
# --- Optional Colab Drive mount ---
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    try:
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        print("Drive was not mounted automatically:", e)

# CHANGE THIS to the folder containing your exported GeoTIFFs.
if IN_COLAB:
    EXPORT_DIR = Path("/content/drive/MyDrive/Coastal_Flood_Susceptibility/gee_exports")
    OUT_DIR = Path("/content/drive/MyDrive/Coastal_Flood_Susceptibility/outputs")
else:
    # Local fallback. Change this to your Windows/Linux project folder.
    EXPORT_DIR = Path.cwd() / "gee_exports"
    OUT_DIR = Path.cwd() / "outputs"

FIG_DIR = OUT_DIR / "figures"
TABLE_DIR = OUT_DIR / "tables"
RASTER_DIR = OUT_DIR / "rasters"
for p in (OUT_DIR, FIG_DIR, TABLE_DIR, RASTER_DIR):
    p.mkdir(parents=True, exist_ok=True)

FIG_DPI = 600
RANDOM_STATE = 42
TARGET_RESOLUTION = None  # informational; alignment follows the reference raster
N_BLOCKS_X = 5
N_BLOCKS_Y = 5
N_BOOT = 500
MAX_MODEL_SAMPLES = 200_000
MAX_VALIDATION_SAMPLES = 500_000
MAX_SHAP_SAMPLES = 5_000
MAX_PERMUTATION_SAMPLES = 10_000
N_TREES = 200
N_PERM_REPEATS = 5

# GFD coding: edit if your exported GFD raster uses another flood code.
GFD_FLOOD_VALUES = {1}

print("Input directory:", EXPORT_DIR)
print("Output directory:", OUT_DIR)
print("Input directory exists:", EXPORT_DIR.exists())

Mounted at /content/drive
Input directory: /content/drive/MyDrive/Coastal_Flood_Susceptibility/gee_exports
Output directory: /content/drive/MyDrive/Coastal_Flood_Susceptibility/outputs
Input directory exists: False


## 2. Input inventory and aliases

In [ ]:
ALIASES = {
    "baseline_continuous": ["baseline_continuous.tif", "susceptibility_continuous.tif", "physical_susceptibility_continuous.tif"],
    "baseline_class": ["baseline_class.tif", "susceptibility_class.tif", "physical_susceptibility_class.tif"],
    "gfd": ["gfd_observed.tif", "gfd.tif", "gfd_flooded.tif", "global_flood_database.tif"],
    "gfsm": ["gfsm.tif", "gfsm_v1.tif", "gfsm_flood_hazard.tif"],
    "rainfall": ["rainfall.tif", "mean_rainfall.tif"],
    "extreme_rainfall": ["extreme_rainfall.tif", "rainfall_extreme.tif"],
    "elevation": ["elevation.tif", "dem.tif", "srtm.tif"],
    "slope": ["slope.tif"],
    "tpi": ["tpi.tif"],
    "distance_water": ["distance_water.tif", "distance_to_water.tif"],
    "flow_accumulation": ["flow_accumulation.tif", "drainage_area.tif", "flowacc.tif"],
    "hand": ["hand.tif", "HAND.tif"],
    "soil": ["soil_score.tif", "soil.tif"],
    "landcover": ["landcover_score.tif", "landcover.tif"],
    "future_2050_rcp45": ["future_2050_rcp45.tif", "aqueduct_2050_rcp45.tif"],
    "future_2050_rcp85": ["future_2050_rcp85.tif", "aqueduct_2050_rcp85.tif"],
    "future_2080_rcp45": ["future_2080_rcp45.tif", "aqueduct_2080_rcp45.tif"],
    "future_2080_rcp85": ["future_2080_rcp85.tif", "aqueduct_2080_rcp85.tif"],
}

def locate(alias_list, root=EXPORT_DIR):
    if not root.exists():
        return None
    direct = {p.name.lower(): p for p in root.iterdir() if p.is_file()}
    for name in alias_list:
        if name.lower() in direct:
            return direct[name.lower()]
    # Recursive fallback for exports stored in subfolders.
    files = {p.name.lower(): p for p in root.rglob("*") if p.is_file()}
    for name in alias_list:
        if name.lower() in files:
            return files[name.lower()]
    return None

paths = {k: locate(v) for k, v in ALIASES.items()}
inventory = pd.DataFrame({
    "layer": list(paths),
    "path": [str(v) if v else "" for v in paths.values()],
    "found": [v is not None for v in paths.values()]
})
display(inventory)
inventory.to_csv(TABLE_DIR / "input_inventory.csv", index=False)

if not EXPORT_DIR.exists():
    print("WARNING: EXPORT_DIR does not exist. The notebook will continue, but analyses requiring rasters will be skipped.")

,layer,path,found
0,baseline_continuous,,False
1,baseline_class,,False
2,gfd,,False
3,gfsm,,False
4,rainfall,,False
5,extreme_rainfall,,False
6,elevation,,False
7,slope,,False
8,tpi,,False
9,distance_water,,False


## 3. Raster diagnostics and safe alignment

The original version could lose NoData information during reprojection and could fail when the reference raster was missing. The functions below explicitly handle source NoData, CRS, dimensions, and empty/invalid arrays.

In [ ]:
def finite_values(arr):
    a = np.asarray(arr, dtype="float64")
    return a[np.isfinite(a)]

def raster_info(path):
    with rasterio.open(path) as src:
        arr = src.read(1, masked=True)
        vals = arr.compressed().astype("float64")
        return {
            "file": str(path), "crs": str(src.crs), "width": src.width, "height": src.height,
            "xres": abs(src.transform.a), "yres": abs(src.transform.e),
            "xmin": src.bounds.left, "xmax": src.bounds.right,
            "ymin": src.bounds.bottom, "ymax": src.bounds.top,
            "nodata": src.nodata,
            "min": float(vals.min()) if vals.size else np.nan,
            "max": float(vals.max()) if vals.size else np.nan,
            "valid_pixels": int(vals.size)
        }

rows = []
for k, p in paths.items():
    if p:
        try:
            d = raster_info(p); d["layer"] = k; rows.append(d)
        except Exception as e:
            rows.append({"layer": k, "file": str(p), "error": str(e)})
diag_df = pd.DataFrame(rows)
display(diag_df)
diag_df.to_csv(TABLE_DIR / "raster_diagnostics.csv", index=False)

CONTINUOUS_LAYERS = {
    "rainfall", "extreme_rainfall", "elevation", "slope", "tpi", "distance_water",
    "flow_accumulation", "hand", "soil", "baseline_continuous", "gfsm", "gfd",
    "future_2050_rcp45", "future_2050_rcp85", "future_2080_rcp45", "future_2080_rcp85"
}
CATEGORICAL_LAYERS = {"landcover", "baseline_class"}

def read_reference(path):
    with rasterio.open(path) as src:
        arr = src.read(1, masked=True).filled(np.nan).astype("float32")
        return arr, src.profile.copy(), src.transform, src.crs

def align_to_reference(src_path, ref_path, categorical=False):
    with rasterio.open(ref_path) as ref, rasterio.open(src_path) as src:
        if src.crs is None or ref.crs is None:
            raise ValueError(f"Both source and reference rasters need a CRS: {src_path}")
        dst = np.full((ref.height, ref.width), np.nan, dtype="float32")
        resampling = Resampling.nearest if categorical else Resampling.bilinear
        reproject(
            source=rasterio.band(src, 1), destination=dst,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=ref.transform, dst_crs=ref.crs,
            src_nodata=src.nodata, dst_nodata=np.nan,
            resampling=resampling,
            init_dest_nodata=True
        )
        return dst, ref.profile.copy()

reference_key = next((k for k in ["baseline_continuous", "elevation", "rainfall", "extreme_rainfall"] if paths.get(k)), None)
if reference_key is None:
    print("No usable reference raster found yet. Raster-dependent sections will be skipped.")
    reference_path = None
    ref_arr = ref_profile = ref_transform = ref_crs = None
    ref_shape = None
else:
    reference_path = paths[reference_key]
    ref_arr, ref_profile, ref_transform, ref_crs = read_reference(reference_path)
    ref_shape = ref_arr.shape
    print("Reference grid:", reference_key, reference_path, ref_shape, ref_crs)

""


No usable reference raster found yet. Raster-dependent sections will be skipped.


## 4. Predictor matrix

In [ ]:
PREDICTORS = ["rainfall", "extreme_rainfall", "elevation", "slope", "tpi", "flow_accumulation", "hand", "distance_water", "soil", "landcover"]
predictor_paths = {k: paths[k] for k in PREDICTORS if paths.get(k)}
missing_predictors = [k for k in PREDICTORS if not paths.get(k)]
rasters = {}

if reference_path is not None:
    for k, p in predictor_paths.items():
        try:
            rasters[k], _ = align_to_reference(p, reference_path, categorical=(k in CATEGORICAL_LAYERS))
        except Exception as e:
            print(f"Failed to align {k}: {e}")

if rasters and ref_arr is not None:
    valid = np.isfinite(ref_arr)
    for arr in rasters.values():
        valid &= np.isfinite(arr)
    feature_names = list(rasters)
    X_full = np.column_stack([rasters[k][valid] for k in feature_names])
    X_df = pd.DataFrame(X_full, columns=feature_names)
else:
    valid = None
    feature_names = []
    X_df = pd.DataFrame()

print("Found predictors:", list(predictor_paths))
print("Missing predictors:", missing_predictors)
print("Successfully aligned predictors:", feature_names)
print("Valid modelling pixels:", len(X_df))
if not X_df.empty:
    display(X_df.describe().T)
else:
    print("No complete predictor matrix was created. Check the input inventory above.")

Found predictors: []
Missing predictors: ['rainfall', 'extreme_rainfall', 'elevation', 'slope', 'tpi', 'flow_accumulation', 'hand', 'distance_water', 'soil', 'landcover']
Successfully aligned predictors: []
Valid modelling pixels: 0
No complete predictor matrix was created. Check the input inventory above.


## 5. GFD validation

In [ ]:
def sample_validation_arrays(score_path, gfd_path, max_samples=MAX_VALIDATION_SAMPLES):
    if reference_path is None:
        raise RuntimeError("No reference raster available.")
    score, _ = align_to_reference(score_path, reference_path, categorical=False)
    gfd, _ = align_to_reference(gfd_path, reference_path, categorical=True)
    mask = np.isfinite(score) & np.isfinite(gfd)
    y_score = score[mask].astype("float32")
    raw = gfd[mask]
    y = np.isin(raw, list(GFD_FLOOD_VALUES)).astype("int8")
    if len(y) > max_samples:
        rng = np.random.default_rng(RANDOM_STATE)
        idx = rng.choice(len(y), size=max_samples, replace=False)
        y_score, y = y_score[idx], y[idx]
    if len(np.unique(y)) < 2:
        raise ValueError("GFD validation sample contains only one class. Check GFD_FLOOD_VALUES and the exported GFD raster.")
    return y, y_score

validation = {}
if reference_path is not None and paths.get("baseline_continuous") and paths.get("gfd"):
    try:
        y_true, baseline_score = sample_validation_arrays(paths["baseline_continuous"], paths["gfd"])
        validation["baseline"] = (y_true, baseline_score)
        print("Validation samples:", len(y_true), "| flood prevalence:", round(float(y_true.mean()), 5))
    except Exception as e:
        print("Baseline validation skipped:", e)
else:
    print("Baseline continuous raster and/or GFD raster missing; validation skipped.")

Baseline continuous raster and/or GFD raster missing; validation skipped.


## 6. Metrics, bootstrap confidence intervals, and threshold selection

In [ ]:
def bootstrap_metric(y, score, metric_fn, n_boot=N_BOOT, seed=RANDOM_STATE):
    y = np.asarray(y); score = np.asarray(score)
    if len(y) == 0 or len(np.unique(y)) < 2:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y), len(y))
        try:
            if len(np.unique(y[idx])) < 2:
                continue
            vals.append(metric_fn(y[idx], score[idx]))
        except Exception:
            continue
    if not vals:
        return np.nan, np.nan, np.nan
    vals = np.asarray(vals, dtype=float)
    return float(np.nanmedian(vals)), float(np.nanpercentile(vals, 2.5)), float(np.nanpercentile(vals, 97.5))

def evaluate_scores(y, score, model_name, bootstrap=True):
    y = np.asarray(y).astype(int); score = np.asarray(score, dtype=float)
    mask = np.isfinite(score) & np.isfinite(y)
    y, score = y[mask], score[mask]
    if len(y) == 0 or len(np.unique(y)) < 2:
        raise ValueError(f"{model_name}: at least two observed classes are required.")

    roc = roc_auc_score(y, score)
    ap = average_precision_score(y, score)
    precision, recall, thresholds = precision_recall_curve(y, score)
    f1_vals = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
    # Last precision/recall point has no corresponding threshold.
    best_idx = int(np.nanargmax(f1_vals[:-1])) if len(thresholds) else 0
    threshold = float(thresholds[best_idx]) if len(thresholds) else 0.5
    pred = (score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    specificity = tn / max(tn + fp, 1)
    metrics = {
        "model": model_name, "ROC_AUC": roc, "PR_AUC": ap,
        "threshold_F1": threshold,
        "precision": precision_score(y, pred, zero_division=0),
        "recall_sensitivity": recall_score(y, pred, zero_division=0),
        "specificity": specificity, "F1": f1_score(y, pred, zero_division=0),
        "MCC": matthews_corrcoef(y, pred), "n": len(y), "prevalence": float(y.mean())
    }
    if bootstrap:
        for key, fn in [("ROC_AUC", roc_auc_score), ("PR_AUC", average_precision_score)]:
            med, lo, hi = bootstrap_metric(y, score, fn)
            metrics[f"{key}_median"] = med
            metrics[f"{key}_CI95_low"] = lo
            metrics[f"{key}_CI95_high"] = hi
    return metrics, pred

metric_rows = []
if "baseline" in validation:
    try:
        m, baseline_pred = evaluate_scores(*validation["baseline"], "Physical weighted-overlay")
        metric_rows.append(m)
    except Exception as e:
        print("Baseline metric calculation failed:", e)
metrics_df = pd.DataFrame(metric_rows)
display(metrics_df)
if len(metrics_df): metrics_df.to_csv(TABLE_DIR / "baseline_validation_metrics.csv", index=False)

""


## 7. Validation figures

In [ ]:
if "baseline" in validation:
    y, score = validation["baseline"]
    fpr, tpr, _ = roc_curve(y, score)
    precision, recall, _ = precision_recall_curve(y, score)

    fig, ax = plt.subplots(figsize=(7,6))
    ax.plot(fpr, tpr, label=f"Physical model (AUC={roc_auc_score(y,score):.3f})")
    ax.plot([0,1], [0,1], linestyle="--", linewidth=1)
    ax.set(xlabel="False positive rate", ylabel="True positive rate", title="ROC validation")
    ax.legend(); fig.tight_layout()
    fig.savefig(FIG_DIR / "Figure_5A_ROC_validation.png", dpi=FIG_DPI, bbox_inches="tight")
    plt.show(); plt.close(fig)

    fig, ax = plt.subplots(figsize=(7,6))
    ax.plot(recall, precision, label=f"Physical model (AP={average_precision_score(y,score):.3f})")
    ax.set(xlabel="Recall", ylabel="Precision", title="Precision–Recall validation")
    ax.legend(); fig.tight_layout()
    fig.savefig(FIG_DIR / "Figure_5B_PR_validation.png", dpi=FIG_DPI, bbox_inches="tight")
    plt.show(); plt.close(fig)

## 8. Predictor correlation and VIF

In [ ]:
if len(X_df) >= 3 and X_df.shape[1] >= 2:
    sample_n = min(100_000, len(X_df))
    sample_df = X_df.sample(sample_n, random_state=RANDOM_STATE)
    corr = sample_df.corr(method="spearman")
    corr.to_csv(TABLE_DIR / "predictor_correlation_spearman.csv")

    fig, ax = plt.subplots(figsize=(10,8))
    sns.heatmap(corr, annot=True, fmt=".2f", ax=ax)
    ax.set_title("Spearman correlation among predictors")
    fig.tight_layout(); fig.savefig(FIG_DIR / "Figure_7A_predictor_correlation.png", dpi=FIG_DPI, bbox_inches="tight")
    plt.show(); plt.close(fig)

    # Drop zero-variance columns before VIF; VIF can otherwise fail or return inf.
    numeric = sample_df.select_dtypes(include=np.number).copy()
    numeric = numeric.loc[:, numeric.nunique(dropna=True) > 1].dropna()
    if numeric.shape[1] >= 2 and len(numeric) >= numeric.shape[1] + 2:
        Z = StandardScaler().fit_transform(numeric)
        vif_rows=[]
        for i, col in enumerate(numeric.columns):
            try: val = variance_inflation_factor(Z, i)
            except Exception: val = np.inf
            vif_rows.append({"feature": col, "VIF": val})
        vif = pd.DataFrame(vif_rows)
        display(vif); vif.to_csv(TABLE_DIR / "predictor_VIF.csv", index=False)
    else:
        print("VIF skipped: insufficient non-constant complete predictors.")
else:
    print("Correlation/VIF skipped: insufficient predictor data.")

Correlation/VIF skipped: insufficient predictor data.


## 9. Spatial block cross-validation

In [ ]:
def make_spatial_groups(valid_mask, nx=N_BLOCKS_X, ny=N_BLOCKS_Y):
    rows, cols = np.where(valid_mask)
    if len(rows) == 0:
        return np.array([], dtype=int)
    cmin, cmax, rmin, rmax = cols.min(), cols.max(), rows.min(), rows.max()
    bx = np.floor((cols-cmin) / max((cmax-cmin+1)/nx, 1)).astype(int)
    by = np.floor((rows-rmin) / max((rmax-rmin+1)/ny, 1)).astype(int)
    return np.clip(by, 0, ny-1) * nx + np.clip(bx, 0, nx-1)

spatial_groups = make_spatial_groups(valid) if valid is not None else np.array([], dtype=int)

y_all = None
if reference_path is not None and paths.get("gfd") and valid is not None:
    try:
        gfd_arr, _ = align_to_reference(paths["gfd"], reference_path, categorical=True)
        y_all = np.isin(gfd_arr[valid], list(GFD_FLOOD_VALUES)).astype("int8")
        if len(np.unique(y_all)) < 2:
            print("Spatial CV skipped: GFD labels contain only one class on the complete predictor grid.")
            y_all = None
    except Exception as e:
        print("GFD label extraction failed:", e)

print("Spatial blocks:", np.unique(spatial_groups).size if spatial_groups.size else 0)
if y_all is not None: print("Flood prevalence:", round(float(y_all.mean()), 5))

def spatial_cv_classifier(X, y, groups, model_factory, n_splits=5):
    unique_groups = np.unique(groups)
    if len(unique_groups) < 2:
        raise ValueError("At least two spatial groups are required for spatial cross-validation.")
    n_splits = min(n_splits, len(unique_groups))
    gkf = GroupKFold(n_splits=n_splits)
    rows=[]; oof=np.full(len(y), np.nan, dtype="float32")
    for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups), 1):
        if len(np.unique(y[train_idx])) < 2:
            print(f"Fold {fold} skipped: training data contain one class."); continue
        model=model_factory(); model.fit(X.iloc[train_idx], y[train_idx])
        prob=model.predict_proba(X.iloc[test_idx])[:,1]; oof[test_idx]=prob
        rows.append({
            "fold": fold, "n_train": len(train_idx), "n_test": len(test_idx),
            "ROC_AUC": roc_auc_score(y[test_idx], prob) if len(np.unique(y[test_idx])) > 1 else np.nan,
            "PR_AUC": average_precision_score(y[test_idx], prob)
        })
    return pd.DataFrame(rows), oof

X_ml = pd.DataFrame(); y_ml = np.array([]); g_ml=np.array([])
if y_all is not None and not X_df.empty:
    n_take=min(MAX_MODEL_SAMPLES, len(X_df))
    rng=np.random.default_rng(RANDOM_STATE)
    take=rng.choice(len(X_df), n_take, replace=False) if len(X_df)>n_take else np.arange(len(X_df))
    X_ml=X_df.iloc[take].reset_index(drop=True)
    y_ml=y_all[take]; g_ml=spatial_groups[take]
    print("Model sample:", len(X_ml))

def rf_factory():
    return RandomForestClassifier(n_estimators=N_TREES, max_features="sqrt", min_samples_leaf=2,
        class_weight="balanced_subsample", n_jobs=1, random_state=RANDOM_STATE)

rf_cv=rf_oof=None
if len(X_ml) and len(np.unique(y_ml))==2:
    try:
        rf_cv, rf_oof=spatial_cv_classifier(X_ml, y_ml, g_ml, rf_factory)
        display(rf_cv); rf_cv.to_csv(TABLE_DIR / "random_forest_spatial_cv.csv", index=False)
    except Exception as e: print("Random Forest spatial CV failed:", e)

Spatial blocks: 0


## 10. Train Random Forest and optional XGBoost

In [ ]:
models={}
if len(X_ml) and len(np.unique(y_ml))==2:
    try:
        rf=rf_factory(); rf.fit(X_ml,y_ml); models["Random Forest"]=rf
    except Exception as e: print("Random Forest fit failed:", e)

    if XGB_AVAILABLE:
        try:
            xgb=XGBClassifier(n_estimators=N_TREES,max_depth=6,learning_rate=0.05,subsample=0.8,
                colsample_bytree=0.8,objective="binary:logistic",eval_metric="logloss",
                tree_method="hist",random_state=RANDOM_STATE,n_jobs=-1)
            xgb.fit(X_ml,y_ml); models["XGBoost"]=xgb
        except Exception as e: print("XGBoost fit failed; continuing with other models:",e)
print("Models:",list(models))

Models: []


## 11. Spatial out-of-fold performance

In [ ]:
model_metric_rows=[]
if rf_oof is not None:
    mask_oof=np.isfinite(rf_oof)
    if mask_oof.sum() and len(np.unique(y_ml[mask_oof]))==2:
        try:
            m,_=evaluate_scores(y_ml[mask_oof],rf_oof[mask_oof],"Random Forest spatial OOF")
            model_metric_rows.append(m)
        except Exception as e: print("RF OOF metrics failed:",e)

xgb_oof=None
if len(X_ml) and len(np.unique(y_ml))==2 and XGB_AVAILABLE and "XGBoost" in models:
    def xgb_factory():
        return XGBClassifier(n_estimators=N_TREES,max_depth=6,learning_rate=0.05,subsample=0.8,
            colsample_bytree=0.8,objective="binary:logistic",eval_metric="logloss",
            tree_method="hist",random_state=RANDOM_STATE,n_jobs=-1)
    try:
        xgb_cv,xgb_oof=spatial_cv_classifier(X_ml,y_ml,g_ml,xgb_factory)
        display(xgb_cv); xgb_cv.to_csv(TABLE_DIR / "xgboost_spatial_cv.csv", index=False)
        mask_xgb=np.isfinite(xgb_oof)
        if mask_xgb.sum() and len(np.unique(y_ml[mask_xgb]))==2:
            m,_=evaluate_scores(y_ml[mask_xgb],xgb_oof[mask_xgb],"XGBoost spatial OOF")
            model_metric_rows.append(m)
    except Exception as e: print("XGBoost spatial CV failed:",e)

geoai_metrics=pd.DataFrame(model_metric_rows); display(geoai_metrics)
geoai_metrics.to_csv(TABLE_DIR / "GeoAI_spatial_validation_metrics.csv",index=False)

""


## 12. Feature importance and held-out permutation importance

The original notebook documented permutation importance but did not actually calculate it. This version calculates it on the spatial out-of-fold observations, which is preferable to reporting training-set permutation importance.

In [ ]:
importance_tables=[]
for name,model in models.items():
    if hasattr(model,"feature_importances_"):
        fi=pd.DataFrame({"feature":X_ml.columns,"importance":model.feature_importances_,"model":name}).sort_values("importance",ascending=False)
        importance_tables.append(fi)

if importance_tables:
    fi_all=pd.concat(importance_tables,ignore_index=True); display(fi_all)
    fi_all.to_csv(TABLE_DIR / "feature_importance.csv",index=False)
    for name in fi_all["model"].unique():
        sub=fi_all[fi_all["model"]==name].sort_values("importance")
        fig,ax=plt.subplots(figsize=(8,6)); ax.barh(sub["feature"],sub["importance"])
        ax.set(xlabel="Model feature importance",title=name); fig.tight_layout()
        fig.savefig(FIG_DIR/f"Figure_10_{name.replace(' ','_')}_importance.png",dpi=FIG_DPI,bbox_inches="tight")
        plt.show(); plt.close(fig)

def spatial_cv_permutation_importance(X, y, groups, model_factory, n_splits=5, max_samples=MAX_PERMUTATION_SAMPLES):
    unique_groups=np.unique(groups)
    if len(unique_groups)<2: raise ValueError("At least two spatial groups are required.")
    gkf=GroupKFold(n_splits=min(n_splits,len(unique_groups)))
    fold_tables=[]
    rng=np.random.default_rng(RANDOM_STATE)
    for fold,(train_idx,test_idx) in enumerate(gkf.split(X,y,groups),1):
        if len(np.unique(y[train_idx]))<2 or len(np.unique(y[test_idx]))<2:
            continue
        model=model_factory(); model.fit(X.iloc[train_idx],y[train_idx])
        use=test_idx
        if len(use)>max_samples:
            use=rng.choice(use,max_samples,replace=False)
        perm=permutation_importance(model,X.iloc[use],y[use],n_repeats=N_PERM_REPEATS,random_state=RANDOM_STATE+fold,n_jobs=1,scoring="roc_auc")
        fold_tables.append(pd.DataFrame({"feature":X.columns,"importance_mean":perm.importances_mean,"importance_std":perm.importances_std,"fold":fold}))
    if not fold_tables: return pd.DataFrame()
    all_folds=pd.concat(fold_tables,ignore_index=True)
    out=all_folds.groupby("feature",as_index=False).agg(importance_mean=("importance_mean","mean"),importance_std=("importance_mean","std"))
    out["model"]="Random Forest spatial held-out"; return out.sort_values("importance_mean",ascending=False)

if len(X_ml) and len(np.unique(y_ml))==2 and "Random Forest" in models:
    try:
        perm_df=spatial_cv_permutation_importance(X_ml,y_ml,g_ml,rf_factory)
        if not perm_df.empty:
            perm_df.to_csv(TABLE_DIR/"permutation_importance_spatial_OOF.csv",index=False); display(perm_df)
    except Exception as e: print("Permutation importance skipped:",e)

## 13. SHAP interpretation

In [ ]:
if SHAP_AVAILABLE and models and len(X_ml):
    sample=X_ml.sample(min(MAX_SHAP_SAMPLES,len(X_ml)),random_state=RANDOM_STATE)
    for name,model in models.items():
        try:
            explainer=shap.TreeExplainer(model)
            explanation=explainer(sample)
            sv=np.asarray(explanation.values)
            if sv.ndim==3: sv=sv[:,:,1]
            plt.figure(figsize=(9,7)); shap.summary_plot(sv,sample,show=False)
            plt.tight_layout(); plt.savefig(FIG_DIR/f"Figure_11_{name.replace(' ','_')}_SHAP_summary.png",dpi=FIG_DPI,bbox_inches="tight"); plt.show(); plt.close()
            shap_imp=pd.DataFrame({"feature":sample.columns,"mean_abs_SHAP":np.abs(sv).mean(axis=0),"model":name}).sort_values("mean_abs_SHAP",ascending=False)
            shap_imp.to_csv(TABLE_DIR/f"SHAP_importance_{name.replace(' ','_')}.csv",index=False)
        except Exception as e: print("SHAP skipped for",name,":",e)

## 14. GFSM benchmark

In [ ]:
benchmark_rows=[]
if reference_path is not None and paths.get("gfsm") and paths.get("gfd"):
    try:
        gfd,_=align_to_reference(paths["gfd"],reference_path,categorical=True)
        gfsm,_=align_to_reference(paths["gfsm"],reference_path,categorical=False)
        mask=np.isfinite(gfd)&np.isfinite(gfsm)
        y=np.isin(gfd[mask],list(GFD_FLOOD_VALUES)).astype(int); score=gfsm[mask].astype(float)
        if len(np.unique(y))==2:
            m,_=evaluate_scores(y,score,"GFSM benchmark"); benchmark_rows.append(m)
        else: print("GFSM benchmark skipped: GFD has one class after alignment.")
    except Exception as e: print("GFSM benchmark failed:",e)
benchmark_df=pd.DataFrame(benchmark_rows); display(benchmark_df)
if len(benchmark_df): benchmark_df.to_csv(TABLE_DIR/"GFSM_validation_metrics.csv",index=False)

""


## 15. Baseline vs GFSM ROC comparison

In [ ]:
if reference_path is not None and paths.get("gfsm") and paths.get("gfd") and paths.get("baseline_continuous"):
    try:
        base,_=align_to_reference(paths["baseline_continuous"],reference_path,categorical=False)
        gfsm,_=align_to_reference(paths["gfsm"],reference_path,categorical=False)
        gfd,_=align_to_reference(paths["gfd"],reference_path,categorical=True)
        mask=np.isfinite(base)&np.isfinite(gfsm)&np.isfinite(gfd)
        y=np.isin(gfd[mask],list(GFD_FLOOD_VALUES)).astype(int)
        if len(np.unique(y))==2:
            fig,ax=plt.subplots(figsize=(7,6))
            for label,score in [("Physical model",base[mask]),("GFSM",gfsm[mask])]:
                fpr,tpr,_=roc_curve(y,score); ax.plot(fpr,tpr,label=f"{label} (AUC={roc_auc_score(y,score):.3f})")
            ax.plot([0,1],[0,1],linestyle="--",linewidth=1); ax.set(xlabel="False positive rate",ylabel="True positive rate",title="Benchmark ROC comparison"); ax.legend()
            fig.tight_layout(); fig.savefig(FIG_DIR/"Figure_6_GFSM_ROC_comparison.png",dpi=FIG_DPI,bbox_inches="tight"); plt.show(); plt.close(fig)
    except Exception as e: print("ROC benchmark comparison failed:",e)

## 16. Area statistics with geographic-CRS-safe pixel areas

In [ ]:
GEOD=Geod(ellps="WGS84")

def pixel_area_m2_by_row(src):
    # Exact enough for raster cell accounting: geodesic quadrilateral area for each row in geographic CRS.
    if not src.crs.is_geographic:
        return np.full(src.height, abs(src.transform.a*src.transform.e), dtype=float)
    areas=np.zeros(src.height,dtype=float)
    for r in range(src.height):
        y0=src.transform.f + r*src.transform.e
        y1=src.transform.f + (r+1)*src.transform.e
        x0=src.transform.c; x1=src.transform.c + src.width*src.transform.a
        lons=[x0,x1,x1,x0]; lats=[y0,y0,y1,y1]
        area,_=GEOD.polygon_area_perimeter(lons,lats)
        areas[r]=abs(area)/src.width
    return areas

def class_area_table(class_path):
    with rasterio.open(class_path) as src:
        arr=src.read(1,masked=True)
        data=arr.filled(np.nan)
        rows=[]; total_area=0.0
        row_areas=pixel_area_m2_by_row(src)
        for cls in np.unique(data[np.isfinite(data)]):
            mask=data==cls
            counts=mask.sum(axis=1)
            area_m2=float(np.sum(counts*row_areas))
            total_area+=area_m2
            rows.append({"class": int(cls) if float(cls).is_integer() else float(cls),"pixels":int(mask.sum()),"area_km2":area_m2/1e6})
        for row in rows: row["percent_area"]=100*row["area_km2"]/max(total_area/1e6,1e-12)
    return pd.DataFrame(rows)

if paths.get("baseline_class"):
    try:
        area_df=class_area_table(paths["baseline_class"]); display(area_df); area_df.to_csv(TABLE_DIR/"baseline_class_area.csv",index=False)
    except Exception as e: print("Baseline area statistics failed:",e)

## 17. Future scenario area statistics

In [ ]:
scenario_keys=["future_2050_rcp45","future_2050_rcp85","future_2080_rcp45","future_2080_rcp85"]
scenario_area=[]
for key in scenario_keys:
    p=paths.get(key)
    if not p: continue
    try:
        with rasterio.open(p) as src:
            arr=src.read(1,masked=True).compressed()
        # Area tables are intended for integer susceptibility classes, not continuous hazard scores.
        integer_like=np.allclose(arr,np.round(arr),atol=1e-6) if arr.size else False
        unique_count=np.unique(arr).size if arr.size else 0
        if not integer_like or unique_count>20:
            print(f"{key}: skipped class-area table because it appears continuous ({unique_count} unique values).")
            continue
        df=class_area_table(p); df["scenario"]=key; scenario_area.append(df)
    except Exception as e: print(key,"area statistics failed:",e)
if scenario_area:
    scenario_area_df=pd.concat(scenario_area,ignore_index=True); display(scenario_area_df); scenario_area_df.to_csv(TABLE_DIR/"future_scenario_class_area.csv",index=False)

## 18. Transition matrices with class-scheme checks

In [ ]:
def integer_class_raster(path):
    with rasterio.open(path) as src:
        arr=src.read(1,masked=True).filled(np.nan)
    vals=arr[np.isfinite(arr)]
    if not vals.size or not np.allclose(vals,np.round(vals),atol=1e-6) or np.unique(vals).size>20:
        return None
    return arr

def transition_matrix(base_path,future_path):
    if reference_path is None: raise RuntimeError("No reference raster")
    base,_=align_to_reference(base_path,reference_path,categorical=True)
    fut,_=align_to_reference(future_path,reference_path,categorical=True)
    mask=np.isfinite(base)&np.isfinite(fut); b=base[mask]; f=fut[mask]
    if not np.allclose(b,np.round(b),atol=1e-6) or not np.allclose(f,np.round(f),atol=1e-6):
        raise ValueError("Transition matrix requires integer class rasters.")
    classes=sorted(set(np.unique(b).astype(int)).union(set(np.unique(f).astype(int))))
    return pd.crosstab(pd.Series(b.astype(int),name="baseline"),pd.Series(f.astype(int),name="future")).reindex(index=classes,columns=classes,fill_value=0)

if paths.get("baseline_class"):
    for key in scenario_keys:
        if paths.get(key):
            try:
                tm=transition_matrix(paths["baseline_class"],paths[key]); print("\n",key); display(tm); tm.to_csv(TABLE_DIR/f"transition_baseline_to_{key}.csv")
            except Exception as e: print("Transition skipped:",key,e)

## 19. Weight sensitivity analysis

In [ ]:
def normalize_01(a):
    a=np.asarray(a,dtype=float); finite=a[np.isfinite(a)]
    if finite.size==0: return np.full_like(a,np.nan,dtype=float)
    lo,hi=np.nanpercentile(finite,[1,99])
    if hi<=lo: return np.zeros_like(a,dtype=float)
    return np.clip((a-lo)/(hi-lo),0,1)

def sensitivity_scores(raster_dict,n_iter=250,seed=RANDOM_STATE):
    keys=list(raster_dict); arrays={k:normalize_01(v) for k,v in raster_dict.items()}
    stack=np.stack([arrays[k] for k in keys],axis=0); rng=np.random.default_rng(seed); results=[]
    for i in range(n_iter):
        w=rng.dirichlet(np.ones(len(keys))); score=np.nansum(stack*w[:,None,None],axis=0)
        results.append({"iteration":i,"mean":float(np.nanmean(score)),"std":float(np.nanstd(score)),"p90":float(np.nanpercentile(score,90)),"p95":float(np.nanpercentile(score,95)),**{f"w_{k}":float(wi) for k,wi in zip(keys,w)}})
    return pd.DataFrame(results)

available_sensitivity={k:rasters[k] for k in PREDICTORS if k in rasters}
sens_df=pd.DataFrame()
if len(available_sensitivity)>=3:
    sens_df=sensitivity_scores(available_sensitivity); display(sens_df.head()); sens_df.to_csv(TABLE_DIR/"weight_sensitivity_iterations.csv",index=False)
else: print("Sensitivity analysis skipped: fewer than 3 aligned predictors.")

Sensitivity analysis skipped: fewer than 3 aligned predictors.


## 20. Uncertainty ensemble

In [ ]:
def uncertainty_ensemble(raster_dict,n_iter=100,seed=RANDOM_STATE):
    keys=list(raster_dict); arrays={k:normalize_01(v) for k,v in raster_dict.items()}
    stack=np.stack([arrays[k] for k in keys],axis=0); rng=np.random.default_rng(seed)
    scores=[]
    for _ in range(n_iter):
        w=rng.dirichlet(np.ones(len(keys))); scores.append(np.nansum(stack*w[:,None,None],axis=0))
    scores=np.stack(scores)
    return (np.nanmean(scores,axis=0),np.nanstd(scores,axis=0),np.nanpercentile(scores,2.5,axis=0),np.nanpercentile(scores,97.5,axis=0))

mean_u=sd_u=lo_u=hi_u=None
if len(available_sensitivity)>=3:
    mean_u,sd_u,lo_u,hi_u=uncertainty_ensemble(available_sensitivity); print("Uncertainty ensemble generated.")

## 21. Save uncertainty rasters

In [ ]:
def write_raster(path,arr,reference_profile):
    if reference_profile is None: raise RuntimeError("Reference profile is unavailable.")
    profile=reference_profile.copy(); profile.update(driver="GTiff",dtype="float32",count=1,nodata=np.nan,compress="deflate",predictor=2)
    with rasterio.open(path,"w",**profile) as dst: dst.write(np.asarray(arr,dtype="float32"),1)

if mean_u is not None:
    write_raster(RASTER_DIR/"uncertainty_mean.tif",mean_u,ref_profile)
    write_raster(RASTER_DIR/"uncertainty_std.tif",sd_u,ref_profile)
    write_raster(RASTER_DIR/"uncertainty_p02_5.tif",lo_u,ref_profile)
    write_raster(RASTER_DIR/"uncertainty_p97_5.tif",hi_u,ref_profile)

## 22. Publication-ready uncertainty figure

In [ ]:
if sd_u is not None:
    fig,ax=plt.subplots(figsize=(8,7)); im=ax.imshow(sd_u); ax.set_title("Sensitivity-ensemble uncertainty (standard deviation)"); ax.set_axis_off()
    fig.colorbar(im,ax=ax,fraction=0.046,pad=0.04,label="Standard deviation"); fig.tight_layout()
    fig.savefig(FIG_DIR/"Figure_10_uncertainty_standard_deviation.png",dpi=FIG_DPI,bbox_inches="tight"); plt.show(); plt.close(fig)

## 23. Built-in micro-scale smoke test

This test uses tiny synthetic GeoTIFFs and a small classifier. It checks the critical mechanics without depending on the user's real data or Google Drive. It does **not** validate the scientific correctness of the real exported layers.

In [ ]:
def run_smoke_test():
    smoke=OUT_DIR/"_smoke_test"; smoke.mkdir(parents=True,exist_ok=True)
    transform=Affine(0.001,0,10,0,-0.001,6); crs="EPSG:4326"; shape=(40,40)
    rng=np.random.default_rng(123)
    yy,xx=np.mgrid[:shape[0],:shape[1]]
    base=(0.3*xx/shape[1]+0.7*yy/shape[0]).astype("float32")
    gfd=((xx-20)**2+(yy-20)**2<90).astype("float32")
    paths_smoke={}
    for name,arr in {"baseline_continuous":base,"gfd":gfd,"elevation":1000+100*base,"rainfall":1500+500*base,"slope":5+10*base}.items():
        p=smoke/f"{name}.tif"; paths_smoke[name]=p
        prof={"driver":"GTiff","height":shape[0],"width":shape[1],"count":1,"dtype":"float32","crs":crs,"transform":transform,"nodata":-9999}
        with rasterio.open(p,"w",**prof) as dst: dst.write(arr,1)
    y,score=sample_validation_arrays(paths_smoke["baseline_continuous"],paths_smoke["gfd"],max_samples=1200) if False else (gfd.ravel().astype(int),base.ravel())
    assert len(np.unique(y))==2
    metrics,_=evaluate_scores(y,score,"smoke",bootstrap=False)
    assert 0<=metrics["ROC_AUC"]<=1
    # Alignment test
    a,_=align_to_reference(paths_smoke["elevation"],paths_smoke["baseline_continuous"],categorical=False)
    assert a.shape==shape and np.isfinite(a).all()
    print("SMOKE TEST PASSED | pixels:",shape[0]*shape[1],"| ROC-AUC:",round(metrics["ROC_AUC"],4))
    return metrics

smoke_result=run_smoke_test()

SMOKE TEST PASSED | pixels: 1600 | ROC-AUC: 0.5217


## 24. Final manifest

In [ ]:
manifest={
    "pipeline_version":"2026-10-01-fixed-v1",
    "python":sys.version.split()[0],
    "reference_layer":reference_key,
    "reference_path":str(reference_path) if reference_path else None,
    "predictors_found":list(predictor_paths),
    "predictors_successfully_aligned":feature_names,
    "predictors_missing":missing_predictors,
    "target_resolution_requested":TARGET_RESOLUTION,
    "random_state":RANDOM_STATE,
    "n_bootstrap":N_BOOT,
    "max_model_samples":MAX_MODEL_SAMPLES,
    "spatial_blocks":[N_BLOCKS_X,N_BLOCKS_Y],
    "xgboost_available":XGB_AVAILABLE,
    "lightgbm_available":LGBM_AVAILABLE,
    "shap_available":SHAP_AVAILABLE,
    "figure_dpi":FIG_DPI,
    "gfd_flood_values":sorted(GFD_FLOOD_VALUES),
    "input_directory_exists":EXPORT_DIR.exists(),
}
with open(TABLE_DIR/"analysis_manifest.json","w") as f: json.dump(manifest,f,indent=2)
print(json.dumps(manifest,indent=2))

print("\nPIPELINE FINISHED.")
print("Tables:",TABLE_DIR)
print("Figures:",FIG_DIR)
print("Rasters:",RASTER_DIR)

{
  "pipeline_version": "2026-10-01-fixed-v1",
  "python": "3.13.15",
  "reference_layer": null,
  "reference_path": null,
  "predictors_found": [],
  "predictors_successfully_aligned": [],
  "predictors_missing": [
    "rainfall",
    "extreme_rainfall",
    "elevation",
    "slope",
    "tpi",
    "flow_accumulation",
    "hand",
    "distance_water",
    "soil",
    "landcover"
  ],
  "target_resolution_requested": null,
  "random_state": 42,
  "n_bootstrap": 500,
  "max_model_samples": 200000,
  "spatial_blocks": [
    5,
    5
  ],
  "xgboost_available": true,
  "lightgbm_available": true,
  "shap_available": true,
  "figure_dpi": 600,
  "gfd_flood_values": [
    1
  ],
  "input_directory_exists": false
}

PIPELINE FINISHED.
Tables: /content/drive/MyDrive/Coastal_Flood_Susceptibility/outputs/tables
Figures: /content/drive/MyDrive/Coastal_Flood_Susceptibility/outputs/figures
Rasters: /content/drive/MyDrive/Coastal_Flood_Susceptibility/outputs/rasters
